# mine-tuning-model

## 필수 라이브러리 설치

In [3]:
!pip install -U pip

!pip install numpy==1.26.4
!pip install chromadb==0.5.5 langchain==0.2.14 sentence-transformers transformers datasets gradio

In [ ]:
# 충돌뜨면 라이브러리 삭제 용도
!pip uninstall -y jax jaxlib opencv-python opencv-python-headless opencv-contrib-python shap cupy-cuda12x
!pip install numpy==1.26.4
!pip install chromadb==0.5.5 langchain==0.2.14

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip freeze > /content/drive/MyDrive/Colab Notebooks/Github/mine-tuning-model/requirements.txt

## 허깅페이스에서 데이터 가져오기

In [1]:
from datasets import load_dataset

ds = load_dataset("lparkourer10/minecraft-wiki")
print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/371 [00:00<?, ?B/s]

dataset.json:   0%|          | 0.00/49.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87934 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['url', 'question', 'answer'],
        num_rows: 87934
    })
})


In [2]:
# 샘플 3개 출력
for i in range(3):
    print(f"=== 샘플 {i+1} ===")
    print(f"URL   : {ds['train']['url'][i]}")
    print(f"Q     : {ds['train']['question'][i]}")
    print(f"A     : {ds['train']['answer'][i][:200]}")  # 앞 200자만
    print()

=== 샘플 1 ===
URL   : https://minecraft.wiki/w/Minecraft_(franchise)
Q     : What are the main differences between the original Minecraft gameplay and its current version in Windows 10 Edition?
A     : The main difference between the original Minecraft gameplay and its current version in Windows 10 Edition is the addition of new features and game modes, which have been implemented through Microsoft'

=== 샘플 2 ===
URL   : https://minecraft.wiki/w/Minecraft_(franchise)
Q     : Is the Creative mode in Minecraft really as easy to understand and use as it is portrayed in the summary?
A     : No, the creative mode in Minecraft is not entirely easy to understand and use as portrayed in the summary. In fact, some aspects of creative mode can be challenging for new players to grasp. For examp

=== 샘플 3 ===
URL   : https://minecraft.wiki/w/Minecraft_(franchise)
Q     : How does the addition of new biomes, mobs, and game modes in the recent update enhance the overall gameplay experience for player

# 데이터 임베딩


In [3]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print('[입베딩 성공]')
print(f"임베딩 차원: {embedding_model.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[입베딩 성공]
임베딩 차원: 384


/tmp/ipykernel_10053/1680344745.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"임베딩 차원: {embedding_model.get_sentence_embedding_dimension()}")


## Chroma DB 구축

In [12]:
import chromadb

# ChromaDB 초기화
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="minecraft_rag")

# 데이터 5000개 테스트
sample_size = 5000
answers   = ds["train"]["answer"][:sample_size]
questions = ds["train"]["question"][:sample_size]
urls      = ds["train"]["url"][:sample_size]

# 임베딩 생성 및 저장 (배치 처리)batch_size = 500
for i in range(0, sample_size, batch_size):
    batch_answers = answers[i:i+batch_size]
    embeddings = embedding_model.encode(batch_answers, show_progress_bar=True).tolist()

    collection.add(
        documents=batch_answers,
        embeddings=embeddings,
        metadatas=[{"question": q, "url": u} for q, u in zip(questions[i:i+batch_size], urls[i:i+batch_size])],
        ids=[str(j) for j in range(i, i+len(batch_answers))]
    )
    print(f" {min(i+batch_size, sample_size)}/{sample_size} 완료")

print(f"\n{collection.count()}개 저장됨")

AttributeError: `np.float_` was removed in the NumPy 2.0 release. Use `np.float64` instead.